In [1]:
# !pip install --upgrade google-api-python-client google-auth-httplib2 google-auth-oauthlib

     ---------------------------------------- 0.0/13.0 MB ? eta -:--:--
     ----------------- ---------------------- 5.8/13.0 MB 29.4 MB/s eta 0:00:01
     ---------------------- ----------------- 7.3/13.0 MB 18.1 MB/s eta 0:00:01
     ------------------------ --------------- 8.1/13.0 MB 12.9 MB/s eta 0:00:01
     --------------------------- ------------ 8.9/13.0 MB 10.9 MB/s eta 0:00:01
     ----------------------------- ---------- 9.7/13.0 MB 9.4 MB/s eta 0:00:01
     -------------------------------- ------- 10.5/13.0 MB 8.4 MB/s eta 0:00:01
     ---------------------------------- ----- 11.3/13.0 MB 7.7 MB/s eta 0:00:01
     ------------------------------------- -- 12.1/13.0 MB 7.2 MB/s eta 0:00:01
     ---------------------------------------  12.8/13.0 MB 6.8 MB/s eta 0:00:01
     ---------------------------------------- 13.0/13.0 MB 6.6 MB/s eta 0:00:00
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Created wheel for google-

In [25]:
import os
import re
import base64
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from configparser import ConfigParser
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from google.auth.transport.requests import Request
from googleapiclient.discovery import build

# OAuth Scopes
SCOPES = ['https://www.googleapis.com/auth/gmail.compose']

def load_config():
    """Load configuration from INI file with validation"""
    config = ConfigParser()
    config.optionxform = str  # Preserve case sensitivity
    config.read('config.ini', encoding='utf-8')  # Explicit encoding
    
    if not config.read('config.ini'):
        raise ValueError("Could not read config.ini file")
    
    # Verify required sections
    required_sections = ['CLIENTS', 'EMAIL', 'SETTINGS']
    for section in required_sections:
        if not config.has_section(section):
            raise ValueError(f"Missing section [{section}] in config.ini")
    
    return config

def authenticate_gmail():
    """More robust authentication with error handling"""
    creds = None
    token_file = 'token.json'
    
    # 1. Check for existing token
    if os.path.exists(token_file):
        try:
            creds = Credentials.from_authorized_user_file(token_file, SCOPES)
            
            # Check if token needs refresh
            if creds and creds.expired and creds.refresh_token:
                print("Refreshing access token...")
                creds.refresh(Request())
                # Save refreshed token
                with open(token_file, 'w') as token:
                    token.write(creds.to_json())
                return build('gmail', 'v1', credentials=creds)
                
            elif creds and creds.valid:
                return build('gmail', 'v1', credentials=creds)
                
        except Exception as e:
            print(f"Token load/refresh failed: {e}")
            os.remove(token_file)
    
    # 2. Fresh authentication flow
    print("Starting new authentication flow...")
    try:
        flow = InstalledAppFlow.from_client_secrets_file('credentials.json', SCOPES)
        creds = flow.run_local_server(port=0, timeout_seconds=60)
        
        # Save the new token
        with open(token_file, 'w') as token:
            token.write(creds.to_json())
            
        return build('gmail', 'v1', credentials=creds)
        
    except Exception as e:
        print(f"Authentication failed: {e}")
        raise

def parse_transactions(file_path, client_names):
    """Parse transaction instructions from file"""
    transactions = []
    try:
        with open(file_path, 'r') as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                
                # Extract client code and instruction
                match = re.match(
                    r'.*Client ID:\s+(H\d+).*Name:\s+([^,]+),\s+(Please\s+\w+\s+\d+\s+shares of\s+[^\n]+)',
                    line
                )
                if match:
                    client_code = match.group(1)
                    instruction = match.group(3)
                    
                    # Parse instruction
                    action_match = re.match(
                        r'Please\s+(\w+)\s+(\d+)\s+shares of\s+([^\n]+)',
                        instruction
                    )
                    if action_match:
                        transactions.append({
                            'client_code': client_code,
                            'client_name': client_names.get(client_code, match.group(2)),
                            'action': action_match.group(1),
                            'quantity': action_match.group(2),
                            'stock': action_match.group(3).strip()
                        })
    except Exception as e:
        print(f"Error reading transactions: {str(e)}")
    
    return transactions

def get_email_body(config, transaction):
    """Properly format email body with newlines"""
    raw_template = config['EMAIL']['approval_body']
    
    # Handle both literal \n and actual newlines
    if '\\n' in raw_template:
        template = raw_template.replace('\\n', '\n')
    else:
        template = raw_template
    
    return template.format(
        client_names=transaction['client_name'],
        action=transaction['action'].capitalize() + 'ing',
        quantity=transaction['quantity'],
        stock=transaction['stock']
    )
    
def create_transaction_draft(service, config, transaction):
    """Create a draft email for transaction approval"""
    try:
        client_code = transaction['client_code']
        to_email = config['CLIENTS'].get(client_code)
        
        if not to_email:
            print(f"No email found for client {client_code}")
            return
        
        # Create email message
        msg = MIMEMultipart()
        msg['Subject'] = config['EMAIL']['approval_subject']
        msg['From'] = config['EMAIL']['sender']
        msg['To'] = to_email.strip()
        
        if config['EMAIL'].get('carbon_copy'):
            msg['Cc'] = config['EMAIL']['carbon_copy']
        
        # Format email body
        body = get_email_body(config, transaction)
        msg.attach(MIMEText(body))
        
        # Create draft in Gmail
        raw_message = base64.urlsafe_b64encode(msg.as_bytes()).decode()
        draft = {'message': {'raw': raw_message}}
        created_draft = service.users().drafts().create(
            userId='me',
            body=draft
        ).execute()
        print(f"Draft created for {client_code}")
        
    except Exception as error:
        print(f'Error creating draft: {error}')

def main():
    try:
        # Load configuration
        config = load_config()
        print("Configuration loaded successfully!")
        
        # Load client names
        client_names = dict(config['CLIENT_NAMES']) if config.has_section('CLIENT_NAMES') else {}
        
        # Get transaction file path
        transactions_file = config['SETTINGS']['transactions_file']
        print(transactions_file)
        if not os.path.exists(transactions_file):
            raise FileNotFoundError(f"Transaction file not found: {transactions_file}")
        
        # Parse transactions
        transactions = parse_transactions(transactions_file, client_names)
        if not transactions:
            print("No valid transactions found")
            return
        
        # Authenticate with Gmail
        try:
            service = authenticate_gmail()
        except Exception as e:
            print(f"Fatal authentication error: {e}")
            # Optionally: send email notification to admin
            sys.exit(1)
        
        # Create drafts
        for tx in transactions:
            create_transaction_draft(service, config, tx)
            
    except Exception as e:
        print(f"An error occurred: {str(e)}")

if __name__ == '__main__':
    main()

Configuration loaded successfully!
transactions.txt
Draft created for H20741
